In [17]:
from sklearn import datasets
import numpy as np
import pandas as pd
import seaborn as sns

In [15]:
# Load the Penguin dataset
penguins = sns.load_dataset('penguins')

In [16]:
penguins.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [18]:
# Remove the nan values
# Split X and Y
# Split train and test data
# Prepare the pipeline for following steps
    #1. Categorical data encoding(OneHot and Label)
    #2. MinMax Scaling

In [19]:
# Removing Nan values
penguins.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

In [21]:
df = penguins.dropna().reset_index(drop=True)

In [22]:
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
4,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male


In [25]:
# Split training and testing data
X = df.drop('species', axis=1)
y = df['species']

from sklearn.model_selection import train_test_split
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, random_state=42, test_size=0.15, stratify=y)

Xtrain.shape, Xtest.shape

((283, 6), (50, 6))

In [33]:
Xtrain.select_dtypes(exclude='object').head(3)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
246,59.6,17.0,230.0,6050.0
73,42.1,19.1,195.0,4000.0
276,54.3,15.7,231.0,5650.0


In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder



ohe = OneHotEncoder(drop='first', sparse_output=False)
mm_scaler = MinMaxScaler()

column_transformer = ColumnTransformer([
    ('categorical_encoding', ohe, Xtrain.select_dtypes(include='object').columns),
    ('Scaling', mm_scaler, Xtrain.select_dtypes(exclude='object').columns)
],
    remainder='passthrough'
                 )

In [38]:
from sklearn.linear_model import LogisticRegression

lrmodel = LogisticRegression()


In [47]:
# Building a pipeline
pipe = Pipeline([
    ("Feature Transformation", column_transformer),
    ("modeling", lrmodel)
])

In [48]:
pipe.fit(Xtrain, ytrain)

Pipeline(steps=[('Feature Transformation',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categorical_encoding',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  Index(['island', 'sex'], dtype='object')),
                                                 ('Scaling', MinMaxScaler(),
                                                  Index(['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'], dtype='object'))])),
                ('modeling', LogisticRegression())])

In [50]:
prediction = pipe.predict(Xtest)

In [53]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [52]:
print(classification_report(ytest, prediction))

              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        22
   Chinstrap       1.00      1.00      1.00        10
      Gentoo       1.00      1.00      1.00        18

    accuracy                           1.00        50
   macro avg       1.00      1.00      1.00        50
weighted avg       1.00      1.00      1.00        50



In [54]:
confusion_matrix(ytest, prediction)

array([[22,  0,  0],
       [ 0, 10,  0],
       [ 0,  0, 18]])

In [56]:
pipe.score(Xtest, ytest)

1.0

In [58]:
import joblib

# Using joblib saving preprocessing step and model
joblib.dump(pipe, "pipeline.pkl")


['pipeline.pkl']

In [59]:
Xtrain.head()

,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
246,Biscoe,59.6,17.0,230.0,6050.0,Male
73,Torgersen,42.1,19.1,195.0,4000.0,Male
276,Biscoe,54.3,15.7,231.0,5650.0,Male
200,Dream,42.5,17.3,187.0,3350.0,Female
319,Biscoe,43.3,14.0,208.0,4575.0,Female


In [63]:
Xtrain['island'].dtype=='O'

True

In [64]:
for col in Xtrain.columns:
    if Xtrain[col].dtype=='O':
        print(f"{col}->{Xtrain[col].unique().tolist()}")
    else:
        print(f"{col}-> Numerical column.")

island->['Biscoe', 'Torgersen', 'Dream']
bill_length_mm-> Numerical column.
bill_depth_mm-> Numerical column.
flipper_length_mm-> Numerical column.
body_mass_g-> Numerical column.
sex->['Male', 'Female']


In [65]:
model = joblib.load("pipeline.pkl")

In [66]:
"Biscoe  0.0  0.0  0  0  Male".split()

['Biscoe', '0.0', '0.0', '0', '0', 'Male']

In [67]:
Xtrain.head(2)

,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
246,Biscoe,59.6,17.0,230.0,6050.0,Male
73,Torgersen,42.1,19.1,195.0,4000.0,Male


In [69]:
test_data = Xtest.head(1)

In [71]:
test_data.columns=range(len(test_data.columns))

In [75]:
test_data.columns = Xtest.columns

In [77]:
model.predict(test_data)[0]

'Adelie'